# Ophanim dual-mode QLoRA
Run after the 17 October 2026 competition. Attach the private synthetic input bundle. Choose one T4/P100 GPU, Internet on, then Save & Run All. No customer data. Check accelerator quota and 12-hour session limit before starting.

In [ ]:
import os, pathlib, zipfile, json
os.environ['OPHANIM_MODELS_DIR']='/kaggle/working/models'
os.environ['HF_HOME']='/kaggle/working/models/hf-cache'
os.environ['HF_HUB_CACHE']='/kaggle/working/models/hf-cache/hub'
os.environ['OLLAMA_MODELS']='/kaggle/working/models/ollama'
inputs=list(pathlib.Path('/kaggle/input').rglob('ophanim-input.zip'))
assert len(inputs)==1, 'Attach the private dataset containing ophanim-input.zip'
work=pathlib.Path('/kaggle/working/ophanim'); work.mkdir(exist_ok=True)
with zipfile.ZipFile(inputs[0]) as z:
    for item in z.infolist():
        dest=(work/item.filename).resolve()
        assert dest.is_relative_to(work.resolve()), 'Unsafe input archive'
    z.extractall(work)
# Optional gated-model token uses Kaggle Secrets, never print it. Qwen is ungated.
try:
    from kaggle_secrets import UserSecretsClient
    token=UserSecretsClient().get_secret('HF_TOKEN')
    if token: os.environ['HF_TOKEN']=token
except Exception:
    pass


In [ ]:
# CUDA 12.4 retains P100/Pascal support. Do not switch to a CUDA 13 wheel.
%pip install torch==2.6.0 --index-url https://download.pytorch.org/whl/cu124
%pip install -r /kaggle/working/ophanim/ml/requirements-training.txt


In [ ]:
import torch, subprocess, sys
assert torch.cuda.is_available(), 'Select GPU in notebook settings'
print(torch.cuda.get_device_name(0))
VERSION='v1-after-competition'
DATA='/kaggle/working/ophanim/dataset'
# If resuming, attach previous output and set RESUME to its checkpoint directory.
RESUME=None
command=[sys.executable,'/kaggle/working/ophanim/ml/scripts/train.py','--dataset',DATA,'--version',VERSION]
if RESUME: command += ['--resume',RESUME]
subprocess.run(command,check=True)


In [ ]:
adapter=pathlib.Path('/kaggle/working/models/adapters')/VERSION
metrics=json.loads((adapter/'run_metrics.json').read_text())
assert metrics['full_training_completed'], 'Budget stopped training; resume checkpoint in a new session'
subprocess.run([sys.executable,'/kaggle/working/ophanim/ml/scripts/evaluate_gpu.py','--dataset',DATA,'--version',VERSION],check=True)


In [ ]:
# Always package logs and checkpoints, including timed-out runs when possible.
import hashlib
files=[p for p in adapter.rglob('*') if p.is_file()]
checksums={str(p.relative_to(adapter)).replace('\\','/'):hashlib.sha256(p.read_bytes()).hexdigest() for p in files}
(adapter/'checksums.json').write_text(json.dumps(checksums,indent=2))
with zipfile.ZipFile('/kaggle/working/ophanim-output.zip','w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in adapter.rglob('*'):
        if p.is_file(): z.write(p,str(p.relative_to(adapter)))
print('Download ophanim-output.zip from Output. Accepted:',json.loads((adapter/'comparison.json').read_text())['accepted'])
